In [1]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM

class BitNetSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, W):
        gamma = W.abs().mean()
        W_quant = torch.clamp(torch.round(W / (gamma + 1e-5)), min=-1, max=1)
        return W_quant

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

class BitLinear(nn.Linear):
    def forward(self, x):
        w_quant = BitNetSTE.apply(self.weight)
        return nn.functional.linear(x, w_quant, self.bias)

def perform_surgery(model):
    for name, module in model.named_modules():
        if "mlp.c_fc" in name or "mlp.c_proj" in name:
            in_features = module.weight.shape[0]
            out_features = module.weight.shape[1]
            new_layer = BitLinear(in_features, out_features, bias=False)
            with torch.no_grad():
                new_layer.weight.copy_(module.weight.t())

            parent_name = name.rsplit('.', 1)[0]
            child_name = name.rsplit('.', 1)[1]
            parent = dict(model.named_modules())[parent_name]
            setattr(parent, child_name, new_layer)
    print("Surgery Successful! GPT-2 is now 1.58-bit.")

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained("gpt2")
perform_surgery(model)

print("\nVerification:")
print(model.transformer.h[0].mlp.c_fc)

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Surgery Successful! GPT-2 is now 1.58-bit.

Verification:
BitLinear(in_features=768, out_features=3072, bias=False)


In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset
from transformers import GPT2Tokenizer
import torch
import torch.optim as optim
import gc # Garbage Collector to help clear memory

# 1. Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("Downloading training data (WikiText)...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# We grab 40 decent-sized sentences
texts = [text for text in dataset['text'][:200] if len(text.strip()) > 50][:40]

optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# --- THE OOM FIX: MICRO-BATCHING ---
batch_size = 4  # Only process 4 sentences at a time!
epochs = 3      # Let's just do 3 full passes of the 40 sentences to save time

print("\nStarting 1.58-bit Fine-Tuning with Micro-Batching...")
model.train()

for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1}/{epochs} ---")
    epoch_loss = 0

    # Slice our 40 sentences into chunks of 4
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=128, truncation=True, padding="max_length")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # We must clear the gradients before every chunk
        optimizer.zero_grad()

        # FORWARD PASS
        outputs = model(input_ids=inputs['input_ids'],
                        attention_mask=inputs['attention_mask'],
                        labels=inputs['input_ids'])
        loss = outputs.loss

        # BACKWARD PASS (Your BitNetSTE at work!)
        loss.backward()

        # UPDATE WEIGHTS
        optimizer.step()

        epoch_loss += loss.item()

        # Clean up the GPU memory after every tiny batch
        del inputs, outputs, loss
        torch.cuda.empty_cache()
        gc.collect()

        print(f"  Processed batch {i//batch_size + 1}... ")

    # Calculate average loss for this epoch
    avg_loss = epoch_loss / (len(texts) / batch_size)
    print(f">> Average Loss for Epoch {epoch+1}: {avg_loss:.4f}")

print("\nTraining complete! Your BitNet model is alive!")


Starting 1.58-bit Fine-Tuning with Micro-Batching...

--- Epoch 1/3 ---


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Processed batch 1... 
  Processed batch 2... 
  Processed batch 3... 
  Processed batch 4... 
  Processed batch 5... 
  Processed batch 6... 
  Processed batch 7... 
  Processed batch 8... 
  Processed batch 9... 
  Processed batch 10... 
>> Average Loss for Epoch 1: 8.6477

--- Epoch 2/3 ---
  Processed batch 1... 
  Processed batch 2... 
  Processed batch 3... 
  Processed batch 4... 
  Processed batch 5... 
  Processed batch 6... 
  Processed batch 7... 
  Processed batch 8... 
  Processed batch 9... 
  Processed batch 10... 
>> Average Loss for Epoch 2: 6.7597

--- Epoch 3/3 ---
  Processed batch 1... 
  Processed batch 2... 
  Processed batch 3... 
  Processed batch 4... 
  Processed batch 5... 
  Processed batch 6... 
  Processed batch 7... 
  Processed batch 8... 
  Processed batch 9... 
  Processed batch 10... 
>> Average Loss for Epoch 3: 5.9542

Training complete! Your BitNet model is alive!


In [5]:
from datasets import load_dataset
from transformers import GPT2Tokenizer
import torch
import torch.optim as optim
import gc
import time
import os

# --- 1. SETUP & DATA PREP ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Assuming 'model' is already loaded and modified with BitLinear from the previous steps
model.to(device)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("Downloading full training data (WikiText)...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# We drop the [:40] slice and grab ALL decent sentences (roughly 20,000+)
print("Filtering dataset...")
texts = [text for text in dataset['text'] if len(text.strip()) > 50]
print(f"Total sentences ready for training: {len(texts)}")

optimizer = optim.AdamW(model.parameters(), lr=5e-5)
batch_size = 4

# --- 2. SHOWCASE LOGIC (The Evolution Function) ---
def showcase_evolution(step, elapsed_time):
    """Pauses training to let the model generate text so we can watch it learn."""
    model.eval() # Turn off training mode
    prompt = "The future of artificial intelligence is"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    print(f"\n[{elapsed_time:.1f} Minutes] - Model Evolution Showcase (Step {step}):")
    print("-" * 50)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=35, pad_token_id=tokenizer.eos_token_id, do_sample=True, temperature=0.7)

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt:   {prompt}")
    print(f"Response: {generated_text.replace(prompt, '').strip()}")
    print("-" * 50 + "\n")

    model.train() # Turn training back on!

    # Clean up generation memory
    del inputs, outputs
    torch.cuda.empty_cache()

# --- 3. THE 2-HOUR TIME-BOUND LOOP ---
max_duration_seconds = 2 * 60 * 60  # 2 Hours
start_time = time.time()

# Timers for our showcase and save features
last_showcase_time = start_time
last_save_time = start_time
showcase_interval = 15 * 60  # Every 15 minutes
save_interval = 30 * 60      # Every 30 minutes

checkpoint_dir = "./bitnet_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"\n🚀 Starting 2-Hour 1.58-bit Fine-Tuning Run...")
model.train()

step = 0
running_loss = 0.0

# We use a 'while' loop based on time instead of a 'for' loop based on epochs
while (time.time() - start_time) < max_duration_seconds:

    for i in range(0, len(texts), batch_size):
        # Check time before every batch
        current_time = time.time()
        if (current_time - start_time) >= max_duration_seconds:
            break # 2 hours are up!

        # --- EVENT: 15-Minute Showcase ---
        if (current_time - last_showcase_time) >= showcase_interval:
            showcase_evolution(step, (current_time - start_time) / 60)
            last_showcase_time = current_time

        # --- EVENT: 30-Minute Checkpoint Save ---
        if (current_time - last_save_time) >= save_interval:
            save_path = f"{checkpoint_dir}/model_min_{int((current_time - start_time)/60)}"
            model.save_pretrained(save_path)
            print(f"💾 Checkpoint saved to {save_path}")
            last_save_time = current_time

        # --- STANDARD TRAINING ---
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=128, truncation=True, padding="max_length")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        optimizer.zero_grad()

        outputs = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], labels=inputs['input_ids'])
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        step += 1

        # Memory cleanup
        del inputs, outputs, loss
        torch.cuda.empty_cache()
        gc.collect()

        # Print a simple progress log every 50 steps
        if step % 50 == 0:
            avg_loss = running_loss / 50
            elapsed_mins = (time.time() - start_time) / 60
            print(f"Step {step} | Time: {elapsed_mins:.1f}m / 120.0m | Avg Loss: {avg_loss:.4f}")
            running_loss = 0.0

print("\n🎉 2-Hour Training Complete! Saving final model...")
model.save_pretrained("./bitnet_final_2hr")
tokenizer.save_pretrained("./bitnet_final_2hr")
print("✅ Final model saved to './bitnet_final_2hr'")

Filtering dataset...
Total sentences ready for training: 16184

🚀 Starting 2-Hour 1.58-bit Fine-Tuning Run...
Step 50 | Time: 0.5m / 120.0m | Avg Loss: 5.2505
Step 100 | Time: 0.9m / 120.0m | Avg Loss: 6.3251
Step 150 | Time: 1.2m / 120.0m | Avg Loss: 5.3036
Step 200 | Time: 1.6m / 120.0m | Avg Loss: 5.4012
Step 250 | Time: 2.0m / 120.0m | Avg Loss: 5.8042
Step 300 | Time: 2.4m / 120.0m | Avg Loss: 5.4309
Step 350 | Time: 2.7m / 120.0m | Avg Loss: 4.7163
Step 400 | Time: 3.1m / 120.0m | Avg Loss: 5.1378
Step 450 | Time: 3.4m / 120.0m | Avg Loss: 4.7020
Step 500 | Time: 3.8m / 120.0m | Avg Loss: 4.2473
Step 550 | Time: 4.1m / 120.0m | Avg Loss: 3.0547
Step 600 | Time: 4.5m / 120.0m | Avg Loss: 5.0387
Step 650 | Time: 4.8m / 120.0m | Avg Loss: 3.5206
Step 700 | Time: 5.2m / 120.0m | Avg Loss: 5.0945
Step 750 | Time: 5.5m / 120.0m | Avg Loss: 5.3461
Step 800 | Time: 5.9m / 120.0m | Avg Loss: 4.8723
Step 850 | Time: 6.3m / 120.0m | Avg Loss: 4.3778
Step 900 | Time: 6.6m / 120.0m | Avg Loss

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Checkpoint saved to ./bitnet_checkpoints/model_min_30

[30.1 Minutes] - Model Evolution Showcase (Step 4168):
--------------------------------------------------
Prompt:   The future of artificial intelligence is
Response: the general and the number of the most common " , the world . The largest scale , the general population of the first time , the first time
--------------------------------------------------

Step 4200 | Time: 30.4m / 120.0m | Avg Loss: 3.9115
Step 4250 | Time: 30.7m / 120.0m | Avg Loss: 4.0558
Step 4300 | Time: 31.1m / 120.0m | Avg Loss: 4.5689
Step 4350 | Time: 31.5m / 120.0m | Avg Loss: 4.2222
Step 4400 | Time: 31.8m / 120.0m | Avg Loss: 3.7954
Step 4450 | Time: 32.2m / 120.0m | Avg Loss: 4.0536
Step 4500 | Time: 32.6m / 120.0m | Avg Loss: 3.7300
Step 4550 | Time: 32.9m / 120.0m | Avg Loss: 3.4437
Step 4600 | Time: 33.3m / 120.0m | Avg Loss: 2.4798
Step 4650 | Time: 33.6m / 120.0m | Avg Loss: 4.4386
Step 4700 | Time: 34.0m / 120.0m | Avg Loss: 2.8407
Step 4750 | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Checkpoint saved to ./bitnet_checkpoints/model_min_60

[60.1 Minutes] - Model Evolution Showcase (Step 8377):
--------------------------------------------------
Prompt:   The future of artificial intelligence is
Response: the concept of human life . The German government has been the sole source of the Soviet Union , and the main force in which the Soviets . In
--------------------------------------------------

Step 8400 | Time: 60.3m / 120.0m | Avg Loss: 3.9541
Step 8450 | Time: 60.7m / 120.0m | Avg Loss: 3.5102
Step 8500 | Time: 61.0m / 120.0m | Avg Loss: 3.7130
Step 8550 | Time: 61.4m / 120.0m | Avg Loss: 3.4063
Step 8600 | Time: 61.7m / 120.0m | Avg Loss: 3.0402
Step 8650 | Time: 62.1m / 120.0m | Avg Loss: 2.4579
Step 8700 | Time: 62.4m / 120.0m | Avg Loss: 4.1112
Step 8750 | Time: 62.8m / 120.0m | Avg Loss: 2.5384
Step 8800 | Time: 63.1m / 120.0m | Avg Loss: 3.9398
Step 8850 | Time: 63.5m / 120.0m | Avg Loss: 4.2644
Step 8900 | Time: 63.8m / 120.0m | Avg Loss: 3.7449
Step 8950 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Checkpoint saved to ./bitnet_checkpoints/model_min_90

[90.1 Minutes] - Model Evolution Showcase (Step 12686):
--------------------------------------------------
Prompt:   The future of artificial intelligence is
Response: the most definitive and valuable tool of the world . The Israeli war is an important source of Israel . A number of natural resources are the largest source
--------------------------------------------------

Step 12700 | Time: 90.2m / 120.0m | Avg Loss: 2.5325
Step 12750 | Time: 90.6m / 120.0m | Avg Loss: 3.8058
Step 12800 | Time: 90.9m / 120.0m | Avg Loss: 2.3802
Step 12850 | Time: 91.3m / 120.0m | Avg Loss: 3.7186
Step 12900 | Time: 91.6m / 120.0m | Avg Loss: 4.0733
Step 12950 | Time: 92.0m / 120.0m | Avg Loss: 3.5225
Step 13000 | Time: 92.3m / 120.0m | Avg Loss: 3.2637
Step 13050 | Time: 92.7m / 120.0m | Avg Loss: 3.6179
Step 13100 | Time: 93.0m / 120.0m | Avg Loss: 3.6214
Step 13150 | Time: 93.4m / 120.0m | Avg Loss: 3.5395
Step 13200 | Time: 93.7m / 120.0m | 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final model saved to './bitnet_final_2hr'


In [6]:
# 1. Put the model in "evaluation" mode (turns off the training gradients)
model.eval()

# 2. Give it a starting prompt
prompt_text = "The future of artificial intelligence is"
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

print(f"Prompt: {prompt_text}")
print("-" * 40)
print("Generating response...\n")

# 3. Generate! (We limit it to 30 tokens so it doesn't ramble)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=30,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,      # Let it be a little creative
        temperature=0.7      # Controls the 'craziness' of the output
    )

# 4. Decode the numbers back into English words
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Prompt: The future of artificial intelligence is
----------------------------------------
Generating response...

The future of artificial intelligence is the most powerful and influential and influential than any of the most important computer , and the computer is the best known for its


In [7]:
import os

# 1. Create a folder to hold your custom model
save_directory = "./my-first-bitnet"
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# 2. Save the Model Weights and the Dictionary (Tokenizer)
print("Saving model weights to disk...")
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

# 3. Save your custom python code so the model knows how to load!
# (Standard Hugging Face doesn't have 'BitLinear', so we have to save our class)
custom_code = """
import torch
import torch.nn as nn
from transformers import PreTrainedModel

class BitNetSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, W):
        gamma = W.abs().mean()
        W_quant = torch.clamp(torch.round(W / (gamma + 1e-5)), min=-1, max=1)
        return W_quant
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

class BitLinear(nn.Linear):
    def forward(self, x):
        w_quant = BitNetSTE.apply(self.weight)
        return nn.functional.linear(x, w_quant, self.bias)
"""

with open(f"{save_directory}/custom_architecture.py", "w") as f:
    f.write(custom_code)

print(f"Success! Your 1.58-bit model is saved in the '{save_directory}' folder.")

Saving model weights to disk...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Your 1.58-bit model is saved in the './my-first-bitnet' folder.


In [8]:
import shutil
from google.colab import files

# 1. The folder you want to download
folder_to_zip = "./bitnet_final_2hr"  # Change to "./my-first-bitnet" if you prefer
zip_filename = "my_custom_bitnet"

print(f"📦 Zipping '{folder_to_zip}' into {zip_filename}.zip...")
print("This might take a minute or two because model files are large...")

# 2. Compress the folder
shutil.make_archive(zip_filename, 'zip', folder_to_zip)

# 3. Trigger the browser download
print("⬇️ Triggering download to your PC...")
files.download(f"{zip_filename}.zip")

📦 Zipping './bitnet_final_2hr' into my_custom_bitnet.zip...
This might take a minute or two because model files are large...
⬇️ Triggering download to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>